# What compiling Python's standard library taught us

### Waspy · Python → WebAssembly, checked against CPython 3.13

<br>

`github.com/anistark/waspy`

In [ ]:
import lab
lab.setup(build_debug=True)   # registers %%compare / %%cpython; builds the compiler driver

## Waspy in one slide

```
program.py ──parse──▶ AST ──lower──▶ IR ──codegen──▶ program.wasm
```

- The output is **WebAssembly**, run by a browser, Node, or `wasmtime`
- There is **no Python interpreter** inside it
- So every module a program imports (`math`, `json`, `datetime`, `logging`, `re`…) has to be **rebuilt by the compiler**

> **The rule:** give CPython's answer, or refuse to compile. Never a guess.

The **IR** ("intermediate representation") is the compiler's own simplified version of the program: a tree of plain statements like *assign*, *call*, *loop*. Code generation turns the IR into WebAssembly.

Every expected value in this talk is what CPython 3.13 prints for the same file. A difference is a bug in Waspy, never "a different opinion".

Every demo is a whole Python program, run **twice**:

| panel | |
|---|---|
| 🟦 **CPython 3.13** | `python3.13 main.py` |
| 🟧 **Waspy → WebAssembly** | compiled by Waspy, run with `wasmtime` |

Verdicts: 🟩 identical · 🟥 differs (lines in red) · 🛑 Waspy refused to build it

In [ ]:
%%compare Hello from both sides
def greet(name: str) -> str:
    return f"Hello, {name}! You have {len(name)} letters."

def main():
    for who in ["Ada", "Grace", "Guido"]:
        print(greet(who))


if __name__ == "__main__":
    main()

## Three parts

1. **What porting taught us** about the standard library itself
2. **Where compile time wins:** mistakes found before the program runs
3. **What's still different**

---

## What porting taught us

## 1 · The standard library is the compiler's hardest test

- `datetime`, `logging`, `re`, `collections` are written **in the compiler's own IR**
- ...then compiled exactly like your code
- Every feature they use (defaults, `super()`, class variables, `Optional`) is one *you* can use

| the module needed | the compiler got wrong |
|---|---|
| a class variable built with a default argument | invalid module |
| `super().__init__(x)` using the parent's defaults | invalid module |
| `Optional[str] == "text"` | compared memory addresses |
| a long `a + b + c + ...` chain | compile time grew exponentially |

All fixed: found by the library, before a user found them.

### The fixes, in one program

In [ ]:
%%compare Defaults, super(), class variables, and Optional strings
from typing import Optional

class Money:
    def __init__(self, amount: int, currency: str = "EUR"):
        self.amount = amount
        self.currency = currency

class Account:
    opening = Money(100)          # class variable, built with a default argument

    def __init__(self, owner: str, nickname: Optional[str] = None):
        self.owner = owner
        self.nickname = nickname

class Savings(Account):
    def __init__(self, owner: str):
        super().__init__(owner)   # relies on the parent's default for nickname

def label(a: Account) -> str:
    if a.nickname == "boss":      # compares text, even though nickname may be None
        return "VIP"
    if a.nickname is None:
        return a.owner
    return a.owner + " aka " + a.nickname

def main():
    print(Account.opening.amount, Account.opening.currency)
    print(label(Account("ada", "boss")))
    print(label(Account("alan", "turing")))
    print(label(Savings("grace")))


if __name__ == "__main__":
    main()

### How big are the ports?

In [ ]:
from pathlib import Path

rows = []
for module in ["datetime", "logging", "re", "collections"]:
    files = sorted((lab.REPO / "src/ir" / module).glob("*.rs"))
    total = sum(len(f.read_text().splitlines()) for f in files)
    rows.append((module, len(files), f"{total:,}", ", ".join(f.name for f in files)))
lab.table(rows, ("module", "files", "lines of Rust", "files"))

Writing the standard library *in the compiler's own terms* pays twice: you get the module, and you get a test of every language feature it touches. A bug the library hits is a bug a user would have hit next.

## 2 · CPython's standard library is mostly Python

- `re` is a **package of Python files**; only the matcher (`_sre`) is C
- `datetime` has a pure-Python twin, `_pydatetime`, giving the same answers
- `logging` is Python, end to end

So porting is mostly *porting Python*. The slow, careful work is the few C engines underneath.

### Looking inside CPython

In [ ]:
%%cpython Where does the re module come from?
import sys, re, re._parser, re._compiler, re._constants, re._casefix
import _pydatetime, _strptime, logging, datetime

def lines(module):
    with open(module.__file__) as f:
        return sum(1 for _ in f)

print("Written in Python:")
for m in [re, re._parser, re._compiler, re._constants, re._casefix, _pydatetime, _strptime, logging]:
    print(f"  {m.__name__:15} {lines(m):5} lines")

print()
print("Written in C, built into the interpreter:")
for name in ["_sre", "_datetime"]:
    where = "built in" if name in sys.builtin_module_names else __import__(name).__file__.split("/")[-1]
    print(f"  {name:15} {where}")

print()
print("datetime.date really comes from:", datetime.date.__module__, "(the C version)")
print("but _pydatetime.date gives the same answers:", _pydatetime.date(2024, 2, 29).isoformat())

### A pattern, as `re._parser` sees it

In [ ]:
%%cpython What re._parser makes of a pattern
import re._parser
tree = re._parser.parse(r"(\d+)-[a-z]?")
tree.dump()

| CPython | | Waspy |
|---|---|---|
| `re/_parser.py` | Python | `src/ir/re/parse.rs`: line-by-line port, **same error messages** |
| `re/_compiler.py` | Python | `src/ir/re/compile.rs` |
| `_sre.c` | C | `src/ir/re/vm.rs`: the backtracking matcher, in IR |

### Fun fact: `strptime` is a regex in disguise

In [ ]:
%%cpython strptime is a regex in disguise
from datetime import datetime
try:
    datetime.strptime("2024 2025", "%Y %Y")
except Exception as e:
    print(type(e).__module__ + "." + type(e).__name__ + ":", e)

CPython turns a `strptime` format like `"%Y-%m-%d"` into a regular expression with one named group per field. That is why reading the same field twice fails with a *regex* error (`re.PatternError: redefinition of group name`), not a date error. *Where compile time wins* shows Waspy catching that one at build time.

## 3 · Dynamic typing hides API quirks

`re.findall` returns a **different type** depending on the pattern's groups:

| groups | returns |
|---|---|
| 0 | `list[str]`, whole matches |
| 1 | `list[str]`, that group |
| 2+ | `list[tuple[str, ...]]` |

Python never notices. A compiler that needs types up front notices immediately.

### `findall` and `split` change shape

In [ ]:
%%compare findall and split change shape with the number of groups
import re

def main():
    text = "ada=1, grace=22, linus=333"
    print(re.findall(r"\w+=\d+", text))        # no groups
    print(re.findall(r"(\w+)=\d+", text))      # one group
    print(re.findall(r"(\w+)=(\d+)", text))    # two groups
    print(re.split(r",\s*", text))             # separators dropped
    print(re.split(r"(,)\s*", text))           # separators kept!


if __name__ == "__main__":
    main()

### Two more shape-shifters

In [ ]:
%%compare A datetime is a date, but never equal to one
from datetime import date, datetime
import logging

def main():
    moment = datetime(2024, 3, 1, 9, 30)
    print(isinstance(moment, date), issubclass(datetime, date))
    print(moment.date() == date(2024, 3, 1))
    print(moment == date(2024, 3, 1))          # False, even on the same day

    log = logging.getLogger("app")
    log.setLevel(20)                           # an int...
    print(log.getEffectiveLevel())
    log.setLevel("WARNING")                    # ...or a name
    print(log.getEffectiveLevel())


if __name__ == "__main__":
    main()

Waspy gives each `findall` call a fixed type by making three pattern classes (`_Pattern0`, `_Pattern1`, `_PatternN`) and picking one when it compiles the pattern, since by then it knows the group count. They all print as `re.Pattern`.

`Logger.setLevel` takes an int *or* a level name: Waspy writes one version of the method per argument type and picks by the argument's type. And `datetime` subclasses `date`, so `isinstance` says yes while `==` always says no.

Design lesson: "the return type depends on the input" is convenient in Python and painful for every tool that reads code.

## 4 · Ignoring case is harder than it looks

`re.IGNORECASE` is **not** "lowercase both sides":

| character | | case-maps to |
|---|---|---|
| `K` | Kelvin sign | `k` |
| `ſ` | long s (old printing) | `S` |
| `ı` | dotless i (Turkish) | `I` |

So `[a-z]` must match the Kelvin sign, though it's nowhere near `a`–`z` in Unicode.

### Letters that change case into other letters

In [ ]:
%%compare Letters that change case into other letters
import re

def main():
    kelvin, long_s, dotless = "K", "ſ", "ı"
    print(kelvin.lower(), long_s.upper(), dotless.upper())
    print(bool(re.match("k", kelvin, re.IGNORECASE)))
    print(bool(re.match("s", long_s, re.IGNORECASE)))
    print(bool(re.match("i", dotless, re.IGNORECASE)))
    print(bool(re.match("[a-z]+", kelvin + "elvin", re.IGNORECASE)))
    print(bool(re.match("[a-z]+", kelvin + "elvin")))       # without the flag


if __name__ == "__main__":
    main()

In [ ]:
%%cpython CPython's table of these special cases
import re._casefix
cases = re._casefix._EXTRA_CASES
print(len(cases), "characters need extra care when case is ignored, for example:")
for code_point in [0x69, 0x73, 0xB5, 0x3B8, 0x3B9]:
    others = ", ".join(f"{chr(c)} (U+{c:04X})" for c in cases[code_point])
    print(f"  {chr(code_point)} also matches {others}")

The Kelvin sign works through ordinary lowercasing. The table is for the harder pairs, where two *different lowercase* letters count as the same letter (`i` and dotless `ı`, `s` and long `ſ`, micro `µ` and Greek `μ`), which no amount of lowercasing discovers. Waspy carries the same table (`EXTRA_CASES` in `src/ir/re/compile.rs`).

Practical tip: compare case-insensitively with `.casefold()`, never `.lower()`.

## 5 · Logging pays for its convenience on every call

- `%(funcName)s` and `%(lineno)d` need to know **where** the log call is
- CPython finds out by **walking the call stack**, for every record it emits
- ...even when your format never prints them

### How much does that cost?

In [ ]:
%%cpython How much of a log call is spent finding the caller?
import logging, time, cProfile, pstats, io

log = logging.getLogger("bench")
log.addHandler(logging.NullHandler())
log.setLevel(logging.INFO)
log.propagate = False

def work(n):
    for i in range(n):
        log.info("order %d done", i)

N = 50_000
t = time.perf_counter(); work(N); with_lookup = time.perf_counter() - t
saved, logging._srcfile = logging._srcfile, None      # skip the caller lookup
t = time.perf_counter(); work(N); without = time.perf_counter() - t
logging._srcfile = saved

print(f"{N:,} records with the caller lookup:    {with_lookup * 1000:6.1f} ms")
print(f"{N:,} records without the caller lookup: {without * 1000:6.1f} ms")
print(f"share of the time spent finding the caller: {(with_lookup - without) / with_lookup:.0%}")

`logging._srcfile = None` is the switch the Logging HOWTO's *Optimization* section documents for turning the lookup off. The point: the line a `log.info(...)` call sits on never changes, yet CPython recomputes it every time. *Where compile time wins* shows Waspy computing it once.

## 6 · `math` gives different digits on different computers

- `+ − × ÷` and `sqrt` must be exact to the last bit
- `sin`, `exp`, `pow`... **don't**: each OS's maths library rounds its own way
- CPython calls the OS's library. Waspy ports **musl**'s, so it's the same everywhere

### 6,400 calls, both sides

In [ ]:
%%compare --lines 6 6,400 maths calls, printed with full precision
import math

def main():
    i = 1
    while i <= 400:
        x = i * 0.737 + i * i * 0.0131
        print("sin", x, repr(math.sin(x)))
        print("cos", x, repr(math.cos(x)))
        print("tan", x, repr(math.tan(x)))
        print("exp", x, repr(math.exp(x / 10.0)))
        print("log", x, repr(math.log(x)))
        print("log10", x, repr(math.log10(x)))
        print("log2", x, repr(math.log2(x)))
        print("atan", x, repr(math.atan(x)))
        print("sinh", x, repr(math.sinh(x / 100.0)))
        print("cosh", x, repr(math.cosh(x / 100.0)))
        print("tanh", x, repr(math.tanh(x / 100.0)))
        print("pow", x, repr(math.pow(x, 0.37)))
        print("asin", x, repr(math.asin(1.0 / (x + 1.0))))
        print("acos", x, repr(math.acos(1.0 / (x + 1.0))))
        print("atan2", x, repr(math.atan2(x, 3.3)))
        print("hypot", x, repr(math.hypot(x, 1.7)))
        i += 1


if __name__ == "__main__":
    main()

In [ ]:
import math

py, ws = lab.last
stats, examples = {}, []
for a, b in zip(py.stdout.splitlines(), ws.stdout.splitlines()):
    name, x, mine = a.split()
    theirs = b.split()[2]
    s = stats.setdefault(name, [0, 0, 0.0])
    s[0] += 1
    if mine != theirs:
        s[1] += 1
        ulps = abs(float(mine) - float(theirs)) / math.ulp(float(mine))
        s[2] = max(s[2], ulps)
        if len(examples) < 6 and name in ("sin", "tan", "exp", "pow"):
            examples.append((f"{name}({x})", mine, theirs, f"{ulps:.0f}"))

rows = sorted(((n, f"{d} of {c}", f"{d / c:.0%}", f"{u:.0f}") for n, (c, d, u) in stats.items()),
              key=lambda r: -int(r[1].split()[0]))
total = sum(d for _, d, _ in stats.values())
lab.callout(f"<b>{total:,} of {sum(c for c, _, _ in stats.values()):,}</b> results differ between this machine's C library and musl, "
            "never by more than the last digit or two.")
lab.table(rows, ("function", "results that differ", "share", "largest gap (ULPs)"))
lab.table(examples, ("call", "CPython (this OS)", "Waspy (musl)", "ULPs apart"))

Neither side is wrong: every difference is a unit or two in the last place (ULP). The point is that the *same Python program prints different digits on different computers*, and almost nobody knows. On Linux the CPython column changes; the Waspy column doesn't.

Tip: never compare `math` results with `==` against stored strings in tests. Use `math.isclose`.

---

## Where compile time wins

Waspy sees the whole program before it runs, so some mistakes CPython only finds *while running* are caught at build time.

## 7 · A broken regex is a build error

- CPython checks a pattern **the first time that line runs**
- Waspy checks **every** pattern while compiling, with CPython's exact message

### Found after the work has started

In [ ]:
%%compare --lines 12 A missing parenthesis, found after the work has started
import re

def main():
    print("loading the order database...")
    print("processing 10,000 orders...")
    order_id = re.compile(r"ORD-(\d{4}")
    print(order_id.match("ORD-1234"))


if __name__ == "__main__":
    main()

### Worse: hiding in a branch no test reaches

In [ ]:
%%compare A bad pattern hiding in a rarely-used branch
import re

def parse_amount(text: str) -> int:
    if text.isdigit():
        return int(text)
    # only reached for input like "12,50": nobody tested that
    euros = re.match(r"(\d+),(\d+", text)
    return int(euros.group(1))

def main():
    print(parse_amount("42"))
    print(parse_amount("1000"))


if __name__ == "__main__":
    main()

In the second demo CPython's run *succeeds*: the tests pass, and the bug waits for the first input like "12,50" in production. A regex in your source is a tiny program in another language; it deserves checking like the rest of the code.

## 8 · A deprecation can be a build error

- Python 3.13 deprecated `count` / `maxsplit` **by position** in `re.sub` / `re.split`
- Why: `re.sub(p, r, s, re.IGNORECASE)` silently passes the flag as the *count*
- CPython prints a warning. Waspy refuses, and says the fix

### Before and after

In [ ]:
%%compare Positional count and maxsplit
import re

def main():
    print(re.sub("a", "-", "banana", 1))
    print(re.split(",", "a,b,c", 1))


if __name__ == "__main__":
    main()

In [ ]:
%%compare The fixed version: pass them by name
import re

def main():
    print(re.sub("a", "-", "banana", count=1))
    print(re.split(",", "a,b,c", maxsplit=1))


if __name__ == "__main__":
    main()

## 9 · Regexes compiled before the program starts

- Compiling a pattern costs far more than using it
- CPython hides that with a cache of **512** patterns
- Past 512, every use compiles again

Waspy builds each pattern into the `.wasm`: no cache, no cliff. The cost moves to build time, paid once by the developer.

### The cache cliff, measured

In [ ]:
%%cpython The cost of compiling a pattern, and of the cache running out
import re, time

def per_call(fn, n):
    t = time.perf_counter()
    for _ in range(n):
        fn()
    return (time.perf_counter() - t) / n * 1e6

pattern = r"(?P<user>[a-z]+)@(?P<host>[a-z.]+)\.(com|org)"
compiled = re.compile(pattern)
print(f"compile it, cache empty:     {per_call(lambda: (re.purge(), re.compile(pattern)), 2000):7.2f} µs")
print(f"compile it again (cached):   {per_call(lambda: re.compile(pattern), 20000):7.2f} µs")
print(f"use it for one match:        {per_call(lambda: compiled.match('ada@math.example.org'), 20000):7.2f} µs")
print()
print(f"the cache holds {re._MAXCACHE} patterns")
for count in (400, 600):
    patterns = [rf"item-{i}-(\d+)" for i in range(count)]
    re.purge()
    for p in patterns:
        re.match(p, "item-5-42")          # first pass fills the cache
    t = time.perf_counter()
    for p in patterns:
        re.match(p, "item-5-42")          # second pass: cached?
    print(f"{count} different patterns, second pass: {(time.perf_counter() - t) * 1000:6.1f} ms")

That is also why Waspy needs the pattern known at compile time: a literal, a module-level constant, or a pattern compiled into a module-level variable. In CPython, the same idea by hand: keep hot patterns as module-level `re.compile(...)` constants.

## 10 · Logging knows the function and line for free

- The compiler writes each log call's **function name and line** into the program
- Nothing walks the stack at run time
- Messages are still formatted **only** if the record is printed

### `funcName` and `lineno`, decided at compile time

In [ ]:
%%compare funcName and lineno, known at compile time
import logging

log = logging.getLogger("shop")

def checkout(total: int):
    log.info("charging %d cents", total)
    if total > 1000:
        log.warning("large order")

def main():
    logging.basicConfig(format="%(levelname)s %(funcName)s:%(lineno)d %(message)s",
                        level=logging.DEBUG)
    checkout(250)
    checkout(5000)


if __name__ == "__main__":
    main()

### Still lazy

In [ ]:
%%compare Formatting only happens for records that are printed
import logging

log = logging.getLogger("lazy")

class Expensive:
    def __str__(self) -> str:
        print("  (building the expensive report)")
        return "big report"

def main():
    logging.basicConfig(format="%(levelname)s %(message)s", level=logging.WARNING)
    report = Expensive()
    log.debug("debug: %s", report)        # off: the report is never built
    log.warning("warning: %s", report)    # on: built once


if __name__ == "__main__":
    main()

In the second demo the `debug` record is switched off, so neither side ever calls `Expensive.__str__` for it. Waspy pre-processes a literal `"... %s"` template when it compiles, and keeps exactly CPython's laziness.

General lesson: facts fixed when the code is written (where a call is, its template) can be worked out once, by the compiler.

## 11 · `json.loads` checks the shape you declared

- In CPython, an annotation on `json.loads(...)` is **only a comment**
- In Waspy it **instructs the parser**
- Wrong-shaped data fails **at the line that parsed it**, not three calls later

### Data that doesn't match the type

In [ ]:
%%compare Data that does not match the declared type
import json

def main():
    counts: list[int] = json.loads('[3, "1", 2]')
    print("parsed:", counts)
    print(sum(counts))


if __name__ == "__main__":
    main()

### The honest flip side

In [ ]:
%%compare Declared float, JSON has an int
import json

def main():
    prices: dict[str, float] = json.loads('{"apple": 1.25, "pear": 2}')
    print(prices)
    print(sum(prices.values()))


if __name__ == "__main__":
    main()

CPython keeps the `int` it found, since it never reads annotations. A `float` place would have stored `2.0`, which prints differently, so Waspy raises `TypeError` at the line that parsed it, the same rule that makes `half(5)` below a build error. This is what libraries like pydantic add on top of CPython; here the compiler does it.

## 12 · You only ship what you import

- A CPython program needs **the interpreter and the whole standard library** installed
- A Waspy program is **one file**, holding only the modules it imports

### Sizes, measured

In [ ]:
import subprocess

programs = {
    "print only":            ("", 'print("hello")'),
    "re.sub":                ("import re", 'print(re.sub(r"\\d+", "#", "call 555 1234"))'),
    "logging.warning":       ("import logging", 'logging.warning("disk %d%% full", 91)'),
    "datetime.date":         ("from datetime import date", 'print(date(2024, 2, 28).isoformat())'),
    "json.dumps":            ("import json", 'print(json.dumps({"a": [1, 2]}))'),
}
rows = []
for name, (imp, body) in programs.items():
    source = f'{imp}\n\ndef main():\n    {body}\n\nif __name__ == "__main__":\n    main()\n'
    run = lab.waspy(source, optimize=True)
    size = f"{run.wasm_bytes / 1024:,.1f} KB" if run.wasm_bytes else (run.refused or "")[:60]
    check = f"import sys; b = set(sys.modules); {imp or 'pass'}; print(len(set(sys.modules) - b))"
    loaded = subprocess.run([lab.PYTHON, "-c", check], capture_output=True, text=True).stdout.strip()
    rows.append((name, size, run.stdout.strip(), f"{loaded} extra modules"))
lab.table(rows, ("program", "Waspy .wasm (optimised)", "its output", "CPython loads for the import"))

A program that prints is a few hundred bytes. `json`, generated straight as WebAssembly for the declared types, is small; the modules written out as IR classes (`re`, `logging`, `datetime`) are bigger, because a module currently comes in **whole**: printing one `date` pulls in `timedelta`, `strftime`, `strptime`... Dropping unused methods is the next step.

## 13 · Strings that are secretly code

Many functions take a little language **as a string**. Each can be checked at build time:

| string | caught at run time today | example |
|---|---|---|
| regex pattern | ✅ already a build error | `"(\d+"` |
| `strptime` format | partly | `"%Y %Y"`, `"%Q"` |
| `open()` mode | not yet | `"rw"` |
| `logging` format | partly | `"%(levelnam)s"` |
| `str.format` | not yet | `"{name:>x}"` |

### Already caught: a field read twice

In [ ]:
%%compare A strptime format that reads the year twice
from datetime import datetime

def main():
    print("parsing...")
    print(datetime.strptime("2024 2025", "%Y %Y"))


if __name__ == "__main__":
    main()

### Not yet: an unknown directive (right answer, but only at run time)

In [ ]:
%%compare An unknown strptime directive: same error, but only at run time
from datetime import datetime

def main():
    print("parsing...")
    print(datetime.strptime("2024-03-01", "%Y-%m-%Q"))


if __name__ == "__main__":
    main()

---

## What's still different

The rule is *CPython's answer or a refusal*. These break it today, and are written down so they get fixed.

## 14 · An `int` passed as a `float` is refused

- CPython ignores the annotation and keeps the `int`: `half(5)` prints `got 5`
- Waspy stores the declared type, so it printed `got 5.0`
- Arithmetic agreed and only printing differed, but that is a different answer, so now it is a **build error**: write `5.0`, or `float(n)`

### `half(5)`

In [ ]:
%%compare An int passed to a float parameter
def half(x: float) -> float:
    print("got", x)
    return x / 2

def main():
    print(half(5))
    print(half(5.0))


if __name__ == "__main__":
    main()

Same root as the json `2`: Waspy stores values as declared, CPython as they are. "Annotations don't change behaviour" is a promise any compiler that *uses* them has to keep deliberately. Waspy keeps it by refusing: a float place cannot remember it was given an int, so the store is a compile error, and the difference left is that Waspy rejects a program CPython runs. Where only the number is used (a comparison, arithmetic, `math.sqrt(4)`, `timedelta(days=1)`) the int still converts, since the answer is the same.

## 15 · Long `+` chains crash the debug compiler

- `a + b + c + ...` parses as `((a + b) + c) + ...`: a tree as deep as the chain
- The compiler walks it by calling itself once per `+`
- A debug build runs out of stack at **~245** terms. Release handles thousands

### 300 strings added together

In [ ]:
terms = " + ".join(f'"w{i}"' for i in range(300))
program = f'''
def main():
    s = {terms}
    print(len(s))

if __name__ == "__main__":
    main()
'''
print(program[:110] + " ... (300 strings in total)")
lab.compare(program, "300 strings added together, debug build of Waspy", build="debug")
lab.compare(program, "The same program, release build of Waspy", build="release")

This is the compiler crashing, not the program. Generated code (templates, big tables, SQL builders) finds this first. The test suite runs on threads with a smaller stack, where the limit is lower still. The fix: walk long chains with a loop.

## 16 · `FileHandler` reopens the file for every record

- CPython opens the log file **once** and keeps it open
- Waspy opens, appends, closes, per record
- Invisible... until something else changes the file

### The usual case: identical

In [ ]:
%%compare A FileHandler, the usual way
import logging

def main():
    log = logging.getLogger("audit")
    log.setLevel(logging.INFO)
    handler = logging.FileHandler("audit.log", mode="w")
    handler.setFormatter(logging.Formatter("%(levelname)s %(message)s"))
    log.addHandler(handler)
    log.info("user ada logged in")
    log.info("user ada bought 3 items")
    log.warning("user ada logged out")
    handler.close()


if __name__ == "__main__":
    main()

### Empty the log while it's open, as log rotation does

In [ ]:
%%compare Emptying the log while it is open (mode="w")
import logging

def main():
    log = logging.getLogger("audit")
    log.setLevel(logging.INFO)
    handler = logging.FileHandler("audit.log", mode="w")
    log.addHandler(handler)
    log.info("first")
    with open("audit.log", "w") as f:     # what log rotation does
        f.write("")
    log.info("second")
    handler.close()


if __name__ == "__main__":
    main()

CPython's open file still remembers it was 6 bytes in, so the next write lands at byte 6 and the gap fills with null bytes, a famous annoyance with `copytruncate` log rotation. With the default `mode="a"` both agree. Waspy's answer here is arguably nicer, but it is a difference, and reopening per record costs time on a busy logger.

## Takeaways

1. **Write the stdlib in the compiler's own terms:** it tests the compiler
2. **Strings that are secretly code** can be checked before anything runs
3. **Facts fixed when code is written** belong to the compiler, not to every run
4. **Declared types can do work:** validation, conversion, better errors
5. **Keep the reference in the loop:** every expected answer here is CPython 3.13's

## Try it

```bash
just notebook          # open this deck as a notebook, run any demo
just notebook-test     # run every cell headless
just notebook-slides   # build a static copy of the slides
```

<br>

**github.com/anistark/waspy**


## Appendix · Everything in one table

| # | finding | today |
|---|---|---|
| 1 | Writing the stdlib in the compiler's IR tests the compiler | ✓ `datetime`, `logging`, `re`, `collections` |
| 2 | CPython's stdlib is mostly Python over a small C core | ✓ ported parser and engine |
| 3 | Return types that depend on the input are API quirks | ✓ one class per group count, one method per argument type |
| 4 | Case-insensitive matching needs a special-case table | ✓ CPython's table |
| 5 | CPython logging walks the stack for every record | measured: about a third of a call |
| 6 | `math` results differ by platform in CPython | ✓ Waspy is the same everywhere |
| 7 | Broken regexes can be build errors | ✓ with CPython's message |
| 8 | Deprecations can be build errors | ✓ `re.sub` / `re.split` |
| 9 | Patterns compiled before the program runs | ✓ no cache, no cliff at 512 |
| 10 | Log call sites known at compile time | ✓ no run-time lookup |
| 11 | Declared types validate parsed JSON | ✓ errors point at the bad value |
| 12 | Ship only what you import | ✓ per module; per method next |
| 13 | String mini-languages checked at build time | partly |
| 14 | `int` into `float` printed `5.0` | ✓ refused at build time |
| 15 | Long `+` chains crash the debug compiler | ✗ known difference |
| 16 | `FileHandler` reopens per record | ✗ known difference |